In [0]:
%run ../source_to_bronze/utils

In [0]:
from pyspark.sql.functions import sum, count, avg, current_date, col

In [0]:
silver_df = spark.table("employee_info.dim_employee")
display(silver_df)

In [0]:
dept_salary_df = (
    silver_df.groupBy("department_name")
             .agg(sum("salary").alias("total_salary"))
             .orderBy(col("total_salary").desc())
             .withColumn("at_load_date", current_date())
)

display(dept_salary_df)

In [0]:
dept_country_emp_count_df = (
    silver_df.groupBy("department_name", "country_name")
             .agg(count("employee_id").alias("employee_count"))
             .withColumn("at_load_date", current_date())
)

display(dept_country_emp_count_df)

In [0]:
dept_country_df = (
    silver_df.select("department_name", "country_name")
             .distinct()
             .withColumn("at_load_date", current_date())
)

display(dept_country_df)

In [0]:
avg_age_df = (
    silver_df.groupBy("department_name")
             .agg(avg("age").alias("avg_age"))
             .withColumn("at_load_date", current_date())
)

display(avg_age_df)

In [0]:
fact_employee_df = silver_df.withColumn("at_load_date", current_date())
display(fact_employee_df)

In [0]:
gold_path = "/Volumes/workspace/default/gold/employee/fact_employee"

In [0]:
from datetime import date

today_dt = str(date.today())

In [0]:
(
    fact_employee_df.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"at_load_date = '{today_dt}'")
    .saveAsTable("employee_info.fact_employee")
)

In [0]:
display(spark.table("employee_info.fact_employee"))